# Fase 3 — Segmentação de Pessoas

Converte as anotações COCO para YOLO-seg, treina o YOLOv8n-seg (classe `person`) e compara visualmente com o detector da Fase 2. Requer os dados do notebook `01_data.ipynb` e os pesos do notebook `02_detection.ipynb` (para a comparação).

## Setup

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow
# --upgrade e necessario: o Colab ja vem com uma versao antiga (2.0.2) do
# pacote kaggle pre-instalada, que nao suporta o token novo nem "python -m kaggle".
!pip install -q --upgrade kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


## 7. Converter COCO para YOLO-seg

Converte os polígonos de anotação do COCO para o formato YOLO-seg.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

DATASET_DIR = ROOT / "data" / "raw" / "coco-person"
ANNOTATIONS_PATH = DATASET_DIR / "instances_person_subset.json"
SPLITS_DIR = ROOT / "data" / "splits" / "coco-person"
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

CLASS_ID = 0  # unica classe: person


def polygon_to_yolo_seg(segmentation, img_w, img_h):
    """Converte uma lista de poligonos COCO (pixel) em linhas YOLO-seg
    normalizadas (0-1). Poligonos com menos de 3 pontos sao descartados."""
    lines = []
    for poly in segmentation:
        if len(poly) < 6:
            continue
        norm = []
        for i in range(0, len(poly), 2):
            x = poly[i] / img_w
            y = poly[i + 1] / img_h
            norm.append(f"{x:.6f}")
            norm.append(f"{y:.6f}")
        lines.append(f"{CLASS_ID} " + " ".join(norm))
    return lines


with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

images_by_id = {img["id"]: img for img in coco["images"]}
anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

LABELS_DIR.mkdir(parents=True, exist_ok=True)

total_labels_written = 0
total_polygons = 0
for img_id, anns in anns_by_image.items():
    img = images_by_id[img_id]
    w, h = img["width"], img["height"]
    stem = Path(img["file_name"]).stem
    label_path = LABELS_DIR / f"{stem}.txt"

    lines = []
    for ann in anns:
        lines.extend(polygon_to_yolo_seg(ann["segmentation"], w, h))

    if lines:
        with open(label_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
        total_labels_written += 1
        total_polygons += len(lines)

print(f"Labels YOLO-seg escritos: {total_labels_written} (poligonos totais: {total_polygons})")

for split in ["train", "val", "test"]:
    manifest = SPLITS_DIR / f"{split}.txt"
    filenames = [
        line.strip() for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()
    ]
    out_path = DATASET_DIR / f"{split}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        for name in filenames:
            f.write((IMAGES_DIR / name).as_posix() + "\n")
    print(f"{split}: {len(filenames)} imagens -> {out_path}")

data_yaml = DATASET_DIR / "data.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    f.write(f"path: {DATASET_DIR.as_posix()}\n")
    f.write("train: train.txt\n")
    f.write("val: val.txt\n")
    f.write("test: test.txt\n")
    f.write("nc: 1\n")
    f.write("names: ['person']\n")

print(f"data.yaml (Ultralytics, segmentacao) escrito em: {data_yaml}")


## 8. Treinar o segmentador (YOLOv8n-seg)

In [ ]:
from ultralytics import YOLO

DATA_YAML = ROOT / "data" / "raw" / "coco-person" / "data.yaml"
PROJECT_DIR = ROOT / "models" / "segmentation"
RUN_NAME = "coco_person_yolov8n_seg_baseline"

# Mesmos hiperparametros de forma da Fase 2, para manter os dois modelos
# comparaveis (docs/relatorio-tecnico.md, secao 3.2).
HYPERPARAMS = dict(
    model="yolov8n-seg.pt",
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="auto",
    seed=42,
    patience=100,
)

model = YOLO(HYPERPARAMS["model"])
model.train(
    data=str(DATA_YAML),
    epochs=HYPERPARAMS["epochs"],
    imgsz=HYPERPARAMS["imgsz"],
    batch=HYPERPARAMS["batch"],
    optimizer=HYPERPARAMS["optimizer"],
    seed=HYPERPARAMS["seed"],
    patience=HYPERPARAMS["patience"],
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)
print(f"Treino concluido. Resultados em: {PROJECT_DIR / RUN_NAME}")


## 9. Comparação visual caixas × máscaras

Roda os dois modelos treinados sobre as mesmas imagens do domínio de canteiro de obra (requer os pesos do detector já treinados em `02_detection.ipynb`).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
SEGMENTATION_WEIGHTS = ROOT / "models" / "segmentation" / "coco_person_yolov8n_seg_baseline" / "weights" / "best.pt"
CSS_IMAGES_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "images"
CSS_LABELS_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "labels"
FIGURES_DIR = ROOT / "reports" / "figures"

PERSON_CLASS_ID_DETECTION = 5  # indice da classe "Person" no dataset de EPIs
NUM_EXAMPLES = 2
CROPS_DIR = FIGURES_DIR / "_crops"

# NOTA: boa parte do dataset Construction Site Safety (versao exportada) e
# composta por imagens-mosaico 2x2 (4 fotos combinadas numa so) - ver
# docs/relatorio-tecnico.md. Para uma comparacao visual legivel, recortamos
# o quadrante onde a pessoa esta centrada. A escolha de imagem/quadrante e
# dinamica (nao usa nomes de arquivo fixos), pois o subconjunto de 350
# imagens da Fase 1 pode variar entre maquinas/sistemas operacionais.


def crop_quadrant(img_path, quadrant, out_path):
    img = Image.open(img_path)
    w, h = img.size
    boxes = {
        "tl": (0, 0, w // 2, h // 2),
        "tr": (w // 2, 0, w, h // 2),
        "bl": (0, h // 2, w // 2, h),
        "br": (w // 2, h // 2, w, h),
    }
    img.crop(boxes[quadrant]).save(out_path)


def pick_quadrant_for_person(label_path):
    """Le o label YOLO e retorna o quadrante (tl/tr/bl/br) onde o centro da
    primeira instancia de Person cai, ou None se nao houver Person."""
    if not label_path.exists():
        return None
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split()
        if int(parts[0]) != PERSON_CLASS_ID_DETECTION:
            continue
        cx, cy = float(parts[1]), float(parts[2])
        return ("t" if cy < 0.5 else "b") + ("l" if cx < 0.5 else "r")
    return None


FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CROPS_DIR.mkdir(parents=True, exist_ok=True)

detector = YOLO(str(DETECTION_WEIGHTS))
segmenter = YOLO(str(SEGMENTATION_WEIGHTS))

images = []
for label_path in sorted(CSS_LABELS_DIR.glob("*.txt")):
    quadrant = pick_quadrant_for_person(label_path)
    if quadrant is None:
        continue
    img_path = CSS_IMAGES_DIR / (label_path.stem + ".jpg")
    if not img_path.exists():
        continue
    out_path = CROPS_DIR / f"{label_path.stem}_{quadrant}.jpg"
    crop_quadrant(img_path, quadrant, out_path)
    images.append(out_path)
    if len(images) >= NUM_EXAMPLES:
        break
print(f"Imagens selecionadas (contem Person): {len(images)}")

for idx, img_path in enumerate(images, start=1):
    det_result = detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    seg_result = segmenter.predict(source=str(img_path), conf=0.25, verbose=False)[0]

    det_img = det_result.plot()
    seg_img = seg_result.plot()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(Image.open(img_path))
    axes[0].set_title("Imagem original")
    axes[0].axis("off")

    axes[1].imshow(det_img[:, :, ::-1])
    axes[1].set_title("Detector Fase 2 (caixas, classe Person)")
    axes[1].axis("off")

    axes[2].imshow(seg_img[:, :, ::-1])
    axes[2].set_title("Segmentador Fase 3 (mascaras, classe person)")
    axes[2].axis("off")

    plt.tight_layout()
    out_path = FIGURES_DIR / f"boxes_vs_masks_example_{idx}.png"
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"Comparacao salva: {out_path}")
